# Conventional CBCT: SIRT reconstruction

SIRT reconstruction of a **measured conventional CBCT scan** (one source on the rotation axis, half-fan offset flat panel) on the exact ASTRA `cone_vec` geometry of the scanner. It is the same geometry `../CBCT` trains NAF on, so the two volumes overlay voxel for voxel.

* **Geometry** comes from `params.py`: SOD / ODD, the half-fan offset `offset_u` and the panel's vertical offset `cbct_offset_v`. The bin factor follows from the pixel width in the `.nii` header.
* **Angles**: frame `i` sits at `START_DEG + i * SPAN_DEG / N`, with `N` the number of frames in the file.
* **Input**: NIfTI with on-disk dims (v, u, frame), values are line integrals `-ln(I / I0)`, row index increasing with +z.
* **Coverage**: the half-fan panel reaches a radius of about 93 mm, and the single cone covers only about 76 mm of z at the rotation axis; slices outside that range are not constrained by the data.

Output: `recon_<name>_SIRT_it<iterations>.nii` in `OUT_DIR`, where `<name>` is the input file name without `.nii` (and without a leading `proj_`). Values are mu in cm^-1, float32, NIfTI dims (x, y, z), voxel width in the header.

## 1. Imports

Run the notebook from this folder (`reconstruction/`), so that `astra_recon.py` and `params.py` are importable.

In [ ]:
import astra
from astra_recon import ConeBeamReconstruction

print(f"ASTRA {astra.__version__}  |  CUDA: {astra.use_cuda()}")

## 2. Parameters (the only cell you normally edit)

* `SPAN_DEG` / `START_DEG` / `ROT_DIR` must match the scan (and the NAF training config's `rot_dir`).
* `IMG_SIZE = (240.0, 240.0, 120.0)` with `VOXEL_SIZE = None` is the NAF training grid at the projection pixel width, so the SIRT and NAF volumes can be compared voxel by voxel.

In [ ]:
PROJ_FILE  = "../input/Real_RANDO_CBCT_04mm/proj_RANDO_CBCT_360proj.nii"
SPAN_DEG   = 360.0            # [deg] frame i sits at START_DEG + i * SPAN_DEG / N
START_DEG  = 0.0              # [deg]
ROT_DIR    = +1
IMG_SIZE   = (240.0, 240.0, 120.0)  # [mm] (x, y, z) — the NAF grid
VOXEL_SIZE = 0.4             # [mm]; None = projection pixel width
ITERATIONS = 300
OUT_DIR    = "./output"
GPU        = None

## 3. Load the projection

Prints the geometry, frames, angles and volume size. **Check the angles line**: it must match the scan.

In [ ]:
rec = ConeBeamReconstruction.from_file(PROJ_FILE, "CBCT", span_deg=SPAN_DEG,
                                       start_deg=START_DEG, rot_dir=ROT_DIR,
                                       img_size=IMG_SIZE, voxel_size=VOXEL_SIZE)

## 4. Reconstruct and show the central slices

SIRT3D_CUDA with a non-negativity constraint. The coronal and sagittal views show +z up.

In [ ]:
rec.run(iterations=ITERATIONS, gpu=GPU)
rec.show_slices()

## 5. Save

Writes `recon_<name>_SIRT_it<iterations>.nii` (mu in cm^-1) to `OUT_DIR`.

In [ ]:
rec.save(OUT_DIR)